# Result store

DuckDB-backed store for batch processing results with resume support.
Generalisation of the old `LLMResultStore`. Works for any operation
(LLM, embeddings, cosine similarity) by accepting a custom column schema.

In [ ]:
#|default_exp utils.result_store

In [ ]:
#|export
from pathlib import Path

import duckdb
import pandas as pd


class ResultStore:
    """DuckDB store for batch processing results with resume/retry support.

    The schema is defined by ``columns``, a dict mapping column names to
    DuckDB type strings.  Two columns are required:

    - An **id column** (default ``"id"``) used for resume and retry logic.
    - An **error column** (default ``"error"``). NULL on success, error
      message string on failure.

    All other columns are user-defined and store the actual results.

    Examples::

        # LLM results
        ResultStore(db_path, {"id": "BIGINT NOT NULL", "data": "VARCHAR NOT NULL", "error": "VARCHAR"})

        # Embeddings
        ResultStore(db_path, {"id": "BIGINT NOT NULL", "embedding": "FLOAT[]", "error": "VARCHAR"})

        # Cosine top-k
        ResultStore(db_path, {"id": "BIGINT NOT NULL", "indices": "INT[]", "scores": "FLOAT[]", "error": "VARCHAR"})

    Usage::

        with ResultStore(db_path, columns) as store:
            remaining = [i for i in all_ids if i not in store.done_ids()]
            ...
            store.insert(chunk_df)
            ...
            n_ok, n_err = store.counts()
    """

    def __init__(self, db_path: str | Path, columns: dict[str, str],
                 table: str = "results", id_col: str = "id", error_col: str = "error",
                 memory_limit: str | None = None):
        self.db_path = str(db_path)
        self.table = table
        self.id_col = id_col
        self.error_col = error_col
        self.conn = duckdb.connect(self.db_path)
        if memory_limit is not None:
            self.conn.execute(f"SET memory_limit = '{memory_limit}'")
        col_defs = ", ".join(f"{name} {typ}" for name, typ in columns.items())
        self.conn.execute(f"CREATE TABLE IF NOT EXISTS {table} ({col_defs})")

    def done_ids(self) -> set:
        """Return set of IDs already in the store."""
        return set(
            self.conn.execute(
                f"SELECT {self.id_col} FROM {self.table}"
            ).fetchnumpy()[self.id_col].tolist()
        )

    def failed_ids(self) -> list:
        """Return list of IDs where error IS NOT NULL."""
        return self.conn.execute(
            f"SELECT {self.id_col} FROM {self.table} WHERE {self.error_col} IS NOT NULL"
        ).fetchnumpy()[self.id_col].tolist()

    def insert(self, df: pd.DataFrame):
        """Insert a DataFrame with columns matching the store schema."""
        self.conn.execute(
            f"INSERT INTO {self.table} BY NAME SELECT * FROM df"
        )

    def delete_ids(self, ids: list):
        """Delete rows by ID."""
        self.conn.execute(
            f"CREATE OR REPLACE TEMP TABLE _del_ids ({self.id_col} BIGINT)"
        )
        self.conn.executemany(
            f"INSERT INTO _del_ids VALUES (?)", [(i,) for i in ids]
        )
        self.conn.execute(
            f"DELETE FROM {self.table} WHERE {self.id_col} IN (SELECT {self.id_col} FROM _del_ids)"
        )

    def counts(self) -> tuple[int, int]:
        """Return (n_success, n_failed)."""
        row = self.conn.execute(f"""
            SELECT
                COUNT(*) FILTER (WHERE {self.error_col} IS NULL),
                COUNT(*) FILTER (WHERE {self.error_col} IS NOT NULL)
            FROM {self.table}
        """).fetchone()
        return row[0], row[1]

    def clear(self):
        """Delete all rows."""
        self.conn.execute(f"DELETE FROM {self.table}")

    def close(self):
        """Close the DuckDB connection and release the reference.

        CHECKPOINT flushes the WAL so that subsequent read-only connections
        can open the database without a 'different configuration' conflict.
        """
        self.conn.execute("CHECKPOINT")
        self.conn.close()
        self.conn = None

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()